In [69]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
import random
import networkx as nx
from sklearn.model_selection import train_test_split
from collections import defaultdict
import itertools

In [71]:
from os import chdir
from pathlib import Path

cwd = Path.cwd()
print(f"CWD: {cwd}")
if cwd.name == "code":
    chdir("..")
print(f"CWD: {Path.cwd()}")

CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main
CWD: /Users/haowen/Documents/Decentral RS/fed-learning-main


In [73]:
def initX(k, R_min=1.0, R_max=5.0):
    scale = np.sqrt((R_max - R_min) / k)
    x = torch.rand(k) * scale
    b = torch.tensor(R_min / 2.0)
    return x, b

def initY(num_items, k, R_min=1.0, R_max=5.0):
    Y = torch.zeros(num_items, k)
    c = torch.zeros(num_items)
    t = torch.zeros(num_items)
    for j in range(num_items):
        Y[j], c[j] = initX(k, R_min, R_max)
    return Y, c, t

class RatingDataset(Dataset):
    def __init__(self, data):
        self.data = torch.tensor(data, dtype=torch.long)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        user, item, rating = self.data[idx]
        return user, item, rating.float()

class MF(nn.Module):
    def __init__(self, num_users, num_items, emb_dim=20):
        super().__init__()
        self.user_emb = nn.Embedding(num_users, emb_dim)
        self.item_emb = nn.Embedding(num_items, emb_dim)
        self.bias = nn.Parameter(torch.zeros(num_items))

    def forward(self, users, items):
        u = self.user_emb(users)
        i = self.item_emb(items)
        return (u * i).sum(1) + self.bias[items]

def compress_model(Y, c, t, compression_ratio):
    num_items = Y.shape[0]
    k = max(1, int(compression_ratio * num_items))
    indices = np.random.choice(num_items, k, replace=False)
    return indices, Y[indices], c[indices], t[indices]

def merge_model(Y, c, t, Y_recv, c_recv, t_recv, indices):
    for i, j in enumerate(indices):
        if t_recv[i] > 0:
            tj, t_new = t[j].item(), t_recv[i].item()
            w = t_new / (tj + t_new) if (tj + t_new) > 0 else 0.5
            t[j] = max(tj, t_new)
            Y[j] = (1 - w) * Y[j] + w * Y_recv[i]
            c[j] = (1 - w) * c[j] + w * c_recv[i]
    return Y, c, t

def train_local(model, dataloader, optimizer, criterion, device):
    model.train()
    for users, items, ratings in dataloader:
        users, items, ratings = users.to(device), items.to(device), ratings.to(device)
        preds = model(users, items)
        loss = criterion(preds, ratings)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

def evaluate(model, dataloader, device):
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for users, items, ratings in dataloader:
            users, items = users.to(device), items.to(device)
            pred = model(users, items)
            preds.extend(pred.cpu().numpy())
            trues.extend(ratings.numpy())
    return np.sqrt(np.mean((np.array(preds) - np.array(trues))**2))

def gossip_learning(num_users, num_items, train_data_dict, test_data_dict, graph, epochs=10, emb_dim=20, lr=0.01, compression_ratio=0.1):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    user_models = {}
    user_Y = {}
    user_c = {}
    user_t = {}
    train_loaders = {}
    test_loaders = {}
    optimizers = {}
    criterion = nn.MSELoss()

    for u in graph.nodes():
        model = MF(num_users, num_items, emb_dim).to(device)
        user_models[u] = model
        optimizers[u] = optim.Adam(model.parameters(), lr=lr)
        Y, c, t = initY(num_items, emb_dim)
        user_Y[u] = Y.clone()
        user_c[u] = c.clone()
        user_t[u] = t.clone()
        model.item_emb.weight.data = Y.clone()
        model.bias.data = c.clone()
        train_set = RatingDataset(train_data_dict[u])
        test_set = RatingDataset(test_data_dict[u])
        train_loaders[u] = DataLoader(train_set, batch_size=64, shuffle=True)
        test_loaders[u] = DataLoader(test_set, batch_size=256)

    for epoch in range(epochs):
        for u in graph.nodes():
            train_local(user_models[u], train_loaders[u], optimizers[u], criterion, device)

        for u in graph.nodes():
            neighbors = list(graph.neighbors(u))
            if not neighbors:
                continue
            partner = random.choice(neighbors)
            indices, Y_s, c_s, t_s = compress_model(user_Y[u], user_c[u], user_t[u], compression_ratio)
            user_Y[partner], user_c[partner], user_t[partner] = merge_model(
                user_Y[partner], user_c[partner], user_t[partner],
                Y_s, c_s, t_s, indices
            )
            user_models[partner].item_emb.weight.data = user_Y[partner]
            user_models[partner].bias.data = user_c[partner]

    rmses = [evaluate(user_models[u], test_loaders[u], device) for u in graph.nodes()]
    return np.mean(rmses)


In [75]:
# Load u.data
df = pd.read_csv("u.data", sep="\t", names=["user_id", "item_id", "rating", "timestamp"])
df = df.drop(columns="timestamp")

user2idx = {u: i for i, u in enumerate(df["user_id"].unique())}
item2idx = {i: j for j, i in enumerate(df["item_id"].unique())}
df["user_id"] = df["user_id"].map(user2idx)
df["item_id"] = df["item_id"].map(item2idx)

num_users = df["user_id"].nunique()
num_items = df["item_id"].nunique()

train_data_dict = defaultdict(list)
test_data_dict = defaultdict(list)

for user_id in range(num_users):
    user_data = df[df["user_id"] == user_id][["user_id", "item_id", "rating"]].values
    if len(user_data) < 5:
        continue
    train, test = train_test_split(user_data, test_size=0.2, random_state=42)
    train_data_dict[user_id] = train
    test_data_dict[user_id] = test

G = nx.erdos_renyi_graph(num_users, 0.05, seed=42)


In [85]:
# Parameter tuning
param_grid = {
    "emb_dim": [10, 20,50,70],
    "lr": [0.01, 0.005,0.001],
    "compression_ratio": [0.3, 0.1],
    "epochs": [100]
}

results = []

for emb_dim, lr, cr, epochs in itertools.product(param_grid["emb_dim"], param_grid["lr"], param_grid["compression_ratio"], param_grid["epochs"]):
    print(f"\nConfig: emb_dim={emb_dim}, lr={lr}, compression={cr}, epochs={epochs}")
    rmse = gossip_learning(num_users, num_items, train_data_dict, test_data_dict, G,
                           epochs=epochs, emb_dim=emb_dim, lr=lr, compression_ratio=cr)
    print(f"Test RMSE: {rmse:.4f}")
    results.append([emb_dim, lr, cr, epochs, rmse])




Config: emb_dim=10, lr=0.01, compression=0.3, epochs=100
Test RMSE: 2.3409

Config: emb_dim=10, lr=0.01, compression=0.1, epochs=100
Test RMSE: 2.3012

Config: emb_dim=10, lr=0.005, compression=0.3, epochs=100
Test RMSE: 2.3990

Config: emb_dim=10, lr=0.005, compression=0.1, epochs=100
Test RMSE: 2.4390

Config: emb_dim=10, lr=0.001, compression=0.3, epochs=100
Test RMSE: 2.9179

Config: emb_dim=10, lr=0.001, compression=0.1, epochs=100
Test RMSE: 2.8809

Config: emb_dim=20, lr=0.01, compression=0.3, epochs=100
Test RMSE: 2.4136

Config: emb_dim=20, lr=0.01, compression=0.1, epochs=100
Test RMSE: 2.4359

Config: emb_dim=20, lr=0.005, compression=0.3, epochs=100
Test RMSE: 2.5106

Config: emb_dim=20, lr=0.005, compression=0.1, epochs=100
Test RMSE: 2.4953

Config: emb_dim=20, lr=0.001, compression=0.3, epochs=100
Test RMSE: 2.7897

Config: emb_dim=20, lr=0.001, compression=0.1, epochs=100
Test RMSE: 2.8408

Config: emb_dim=50, lr=0.01, compression=0.3, epochs=100
Test RMSE: 2.5644

Con

In [87]:
results_df = pd.DataFrame(results, columns=["embedding_dim", "learning_rate", "compression_ratio", "epochs", "test_rmse"])
#results_df.to_csv("gossip_tuning_results.csv", index=False)
#print("\nResults saved to gossip_tuning_results.csv")
results_df

,embedding_dim,learning_rate,compression_ratio,epochs,test_rmse
0,10,0.010,0.3,100,2.340899
1,10,0.010,0.1,100,2.301227
2,10,0.005,0.3,100,2.399029
3,10,0.005,0.1,100,2.438952
4,10,0.001,0.3,100,2.917902
5,10,0.001,0.1,100,2.880893
6,20,0.010,0.3,100,2.413649
7,20,0.010,0.1,100,2.435933
8,20,0.005,0.3,100,2.510645
9,20,0.005,0.1,100,2.495258
